In [1]:
import cv2

In [2]:
face_cap = cv2.CascadeClassifier("haarcascade_frontalcatface.xml")

 # Start camera
video_cap = cv2.VideoCapture(0)

if not video_cap.isOpened():
    print("Error: Camera not opening")
    exit()

while True:
    ret, frame = video_cap.read()
    if not ret:
        break

    # Convert to grayscale
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Improve contrast (helps detection)
    gray = cv2.equalizeHist(gray)

    # Detect faces (correct syntax)
    faces = face_cap.detectMultiScale(gray, 1.1, 4, minSize=(20, 20))

    # Draw rectangle
    for (x, y, w, h) in faces:
        cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)

    # Show output
    cv2.imshow("Face Detection", frame)

    # Press ESC to stop
    if cv2.waitKey(1) == 27:
        break

# Release camera
video_cap.release()
cv2.destroyAllWindows()

    

In [3]:
import sys
!{sys.executable} -m pip install --user face_recognition

In [1]:
import face_recognition 
print("Installed successfully")

C:\ProgramData\anaconda3\Lib\site-packages\face_recognition_models\__init__.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


Installed successfully


In [17]:
import cv2
import face_recognition
import os
import numpy as np
from datetime import datetime

# -----------------------------
# STEP 1: Load Dataset
# -----------------------------
known_encodings = []
known_names = []

for file in os.listdir("dataset"):
    if file.endswith((".jpg", ".png", ".jpeg")):
        img_path = os.path.join("dataset", file)

        img = face_recognition.load_image_file(img_path)
        encodings = face_recognition.face_encodings(img)

        if len(encodings) > 0:
            known_encodings.append(encodings[0])
            known_names.append(file.split(".")[0])

print("Dataset Loaded!")

# -----------------------------
# STEP 2: Attendance Function (NO REPEAT)
# -----------------------------
def mark_attendance(name):
    file_name = "attendance.csv"

    # Create file if not exists
    if not os.path.exists(file_name):
        with open(file_name, "w") as f:
            f.write("Name,Time\n")

    # Read existing names
    with open(file_name, "r") as f:
        data = f.readlines()

    names = [line.split(",")[0].strip() for line in data]

    # Add only if not present
    if name not in names:
        with open(file_name, "a") as f:
            time_now = datetime.now().strftime("%H:%M:%S")
            f.write(f"{name},{time_now}\n")
            print("Attendance marked for:", name)

# -----------------------------
# STEP 3: Start Camera
# -----------------------------
video = cv2.VideoCapture(0)

while True:
    ret, frame = video.read()
    if not ret:
        break

    # Convert to RGB
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Detect faces
    face_locations = face_recognition.face_locations(rgb_frame)
    face_encodings = face_recognition.face_encodings(rgb_frame, face_locations)

    # -----------------------------
    # STEP 4: Match Faces
    # -----------------------------
    for (top, right, bottom, left), face_encoding in zip(face_locations, face_encodings):

        name = "Unknown"

        if len(known_encodings) > 0:
            matches = face_recognition.compare_faces(known_encodings, face_encoding)
            face_distances = face_recognition.face_distance(known_encodings, face_encoding)
            best_match_index = np.argmin(face_distances)

            if matches[best_match_index]:
                name = known_names[best_match_index]

        # -----------------------------
        # STEP 5: Mark Attendance
        # -----------------------------
        if name != "Unknown":
            mark_attendance(name)

        # -----------------------------
        # STEP 6: Draw Box + Name
        # -----------------------------
        cv2.rectangle(frame, (left, top), (right, bottom), (0, 255, 0), 2)
        cv2.putText(frame, name, (left, top - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    # -----------------------------
    # STEP 7: Show Output
    # -----------------------------
    cv2.imshow("Face Recognition Attendance", frame)

    if cv2.waitKey(1) == 27:
        break

# -----------------------------
# STEP 8: Release Camera
# -----------------------------
video.release()
cv2.destroyAllWindows()
 
    



Dataset Loaded!


In [18]:
#day-3
from datetime import datetime
import os

In [19]:
def mark_attendance(name):
    file_name = "attendance.csv"

    if not os.path.exists(file_name):
        with open(file_name, "w") as f:
            f.write("Name,Time\n")

    with open(file_name, "a+") as f:
        f.seek(0)
        data = f.readlines()
        names = [line.split(",")[0] for line in data]

        if True:
            time_now = datetime.now().strftime("%H:%M:%S")
            f.write(f"{name},{time_now}\n")
            print("Attendance marked for:", name)

In [20]:
import cv2
import os

def capture_new_face(name):
    video = cv2.VideoCapture(0)

    print("Press 's' to capture image")

    while True:
        ret, frame = video.read()
        if not ret:
            break

        cv2.imshow("Capture Face", frame)

        key = cv2.waitKey(1)

        # Press 's' to save image
        if key == ord('s'):
            file_path = os.path.join("dataset", f"{name}.jpg")
            cv2.imwrite(file_path, frame)
            print(f"Image saved as {name}.jpg")
            break

        # Press ESC to exit
        elif key == 27:
            break

    video.release()
    cv2.destroyAllWindows()

In [21]:
capture_new_face("virat")

Press 's' to capture image
